# Part 1: Tokenization and Vocabulary

## Notebook 1 — What is Tokenization?

We load a real tokenizer (GPT-2) to understand the core concepts:
- What a tokenizer does (continuous text → discrete tokens)
- Token IDs and special tokens (BOS, EOS, PAD)
- Encoding and decoding round-trip
- How different tokenizers produce different tokenizations


### 1. Load the GPT-2 tokenizer

GPT-2 uses Byte-Pair Encoding (BPE) with a vocabulary of 50,257 tokens. We use HuggingFace's `AutoTokenizer` which auto-detects the tokenizer type.


In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
print(f"Tokenizer class: {type(tokenizer).__name__}")
print(f"Vocab size: {tokenizer.vocab_size}")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizer class: GPT2Tokenizer
Vocab size: 50257


### 2. Tokenize a sentence

The tokenizer splits text into subword units. Let's see what happens to a sentence about robotics.


In [2]:
text = "The robot picks up the red cube and places it on the table."
tokens = tokenizer.tokenize(text)
print(f"Original text: {text}")
print(f"Tokens ({len(tokens)}): {tokens}")


Original text: The robot picks up the red cube and places it on the table.
Tokens (14): ['The', 'Ġrobot', 'Ġpicks', 'Ġup', 'Ġthe', 'Ġred', 'Ġcube', 'Ġand', 'Ġplaces', 'Ġit', 'Ġon', 'Ġthe', 'Ġtable', '.']


### 3. Map tokens to IDs

Each token has a unique integer ID in the vocabulary. The model sees these integers, not the text strings.


In [5]:
token_ids = tokenizer.encode(text)
print(f"Token IDs: {token_ids}")
print(f"Number of tokens: {len(token_ids)}")

# Show token-to-ID mapping
for token, tid in zip(tokens, token_ids):
    print(f"  {token:20s} -> {tid}")


Token IDs: [464, 9379, 11103, 510, 262, 2266, 23441, 290, 4113, 340, 319, 262, 3084, 13]
Number of tokens: 14
  The                  -> 464
  Ġrobot               -> 9379
  Ġpicks               -> 11103
  Ġup                  -> 510
  Ġthe                 -> 262
  Ġred                 -> 2266
  Ġcube                -> 23441
  Ġand                 -> 290
  Ġplaces              -> 4113
  Ġit                  -> 340
  Ġon                  -> 319
  Ġthe                 -> 262
  Ġtable               -> 3084
  .                    -> 13


### 4. Decode back to text

Tokenization is lossy — casing and spacing may differ. But the semantic content is preserved.


In [6]:
decoded = tokenizer.decode(token_ids)
print(f"Decoded: {decoded}")
print(f"Round-trip match: {decoded.strip() == text}")  # usually False due to casing


Decoded: The robot picks up the red cube and places it on the table.
Round-trip match: True


### 5. Special tokens

Language models use special tokens for structure: BOS (beginning of sequence), EOS (end), PAD (padding), UNK (unknown).


In [7]:
print(f"BOS token: {tokenizer.bos_token} -> {tokenizer.bos_token_id}")
print(f"EOS token: {tokenizer.eos_token} -> {tokenizer.eos_token_id}")
print(f"PAD token: {tokenizer.pad_token} -> {tokenizer.pad_token_id}")

# GPT-2 doesn't set pad_token by default
tokenizer.pad_token = tokenizer.eos_token
print(f"PAD token (set to EOS): {tokenizer.pad_token} -> {tokenizer.pad_token_id}")


BOS token: <|endoftext|> -> 50256
EOS token: <|endoftext|> -> 50256
PAD token: None -> None
PAD token (set to EOS): <|endoftext|> -> 50256


### 6. BPE subword tokens

Subword tokenization handles unknown words by splitting them: 'tokenization' becomes 'token' + 'ization'. This is the core compression mechanism.


In [8]:
complex_word = "tokenization"
subwords = tokenizer.tokenize(complex_word)
print(f"{complex_word} -> {subwords}")

# Words not in vocabulary get split
novel_word = "end-effector"
subwords = tokenizer.tokenize(novel_word)
print(f"{novel_word} -> {subwords}")


tokenization -> ['token', 'ization']
end-effector -> ['end', '-', 'effect', 'or']


### Key Takeaway

**Tokenization = discretization + compression.** Any continuous signal that can be tokenized into a discrete vocabulary can be processed by a transformer using next-token prediction. In Part 2, we'll see how robot VLAs handle the reverse problem: mapping continuous actions to tokens.
